# 04 - Agreement, faithfulness and efficiency

*Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2*

This is the notebook that produces the results reported in the thesis. It is self-contained, so it recalculates both sets of importance scores rather than relying on notebooks 02 and 03.

It answers three questions:

- **RQ1 agreement** - do the two methods pick the same heads, and do they match the circuit published by Wang et al. (2023)?
- **RQ2 faithfulness** - if the heads a method picks are switched off, does the behaviour actually break?
- **RQ3 efficiency and sensitivity** - how much faster is the approximation, and do the conclusions hold as the cut-off changes?

Run on Google Colab with an NVIDIA T4 GPU. Random seed fixed at 0.

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon, rankdata
from transformer_lens import HookedTransformer, utils

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
print("GPT-2 small:", n_layers, "layers x", n_heads, "heads | device:", device)

## Dataset

Fifty ordered name pairs across two templates of equal token length, giving 100 sentences. Each has a clean and a corrupted version differing by one token.

In [ ]:
templates = [
    "When{A} and{B} went to the shop,{S} gave a drink to",
    "When{A} and{B} got to the office,{S} sent a letter to",
]

candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]

names = []
for name in candidate_names:
    if model.to_tokens(name, prepend_bos=False).shape[1] == 1:
        names.append(name)

template_lengths = []
for template in templates:
    example = template.format(A=" Mary", B=" John", S=" John")
    template_lengths.append(model.to_tokens(example).shape[1])
assert template_lengths[0] == template_lengths[1], "templates must be the same length in tokens"

all_pairs = []
for name_a in names:
    for name_b in names:
        if name_a != name_b:
            all_pairs.append((name_a, name_b))

random.seed(0)
random.shuffle(all_pairs)
pairs = all_pairs[:50]

clean_prompts = []
corrupted_prompts = []
io_tokens = []
s_tokens = []

for name_a, name_b in pairs:
    for template in templates:
        clean_prompts.append(template.format(A=name_a, B=name_b, S=name_b))
        corrupted_prompts.append(template.format(A=name_a, B=name_b, S=name_a))
        io_tokens.append(model.to_single_token(name_a))
        s_tokens.append(model.to_single_token(name_b))

clean_tokens = model.to_tokens(clean_prompts)
corrupted_tokens = model.to_tokens(corrupted_prompts)
N = len(clean_prompts)

print("name pairs:", len(pairs), "| templates:", len(templates), "| sentences:", N)
print("sequence length:", clean_tokens.shape[1], "tokens")

## Metric and baselines

In [ ]:
def mean_logit_difference_tensor(logits):
    """Result stays a tensor so a gradient can be taken from it."""
    final_logits = logits[:, -1, :]
    total = final_logits[0, io_tokens[0]] - final_logits[0, s_tokens[0]]
    for i in range(1, N):
        total = total + final_logits[i, io_tokens[i]] - final_logits[i, s_tokens[i]]
    return total / N

def mean_logit_difference(logits):
    return mean_logit_difference_tensor(logits).item()

def logit_difference_per_sentence(logits):
    """One value per sentence, needed for the paired statistical tests."""
    final_logits = logits[:, -1, :]
    values = []
    for i in range(N):
        values.append((final_logits[i, io_tokens[i]] - final_logits[i, s_tokens[i]]).item())
    return np.array(values)

In [ ]:
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    CLEAN_BASELINE = mean_logit_difference(clean_logits)
    CORRUPTED_BASELINE = mean_logit_difference(model(corrupted_tokens))

SCALE = CLEAN_BASELINE - CORRUPTED_BASELINE

clean_per_sentence = logit_difference_per_sentence(clean_logits)
n_correct = 0
for value in clean_per_sentence:
    if value > 0:
        n_correct = n_correct + 1

print("clean baseline:    ", round(CLEAN_BASELINE, 3))
print("corrupted baseline:", round(CORRUPTED_BASELINE, 3))
print("scale:             ", round(SCALE, 3))
print("accuracy:          ", round(100 * n_correct / N, 1), "%")

## Method 1 - activation patching

One head at a time, clean activations are inserted into the corrupted run and the recovery is measured.

In [ ]:
head_to_patch = 0   # set before each run

def patch_one_head(z, hook):
    z[:, :, head_to_patch, :] = clean_cache[hook.name][:, :, head_to_patch, :]
    return z

patching_scores = np.zeros((n_layers, n_heads))

with torch.no_grad():
    for layer in range(n_layers):
        hook_name = utils.get_act_name("z", layer)
        for head in range(n_heads):
            head_to_patch = head
            patched_logits = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(hook_name, patch_one_head)])
            patched_value = mean_logit_difference(patched_logits)
            patching_scores[layer, head] = (patched_value - CORRUPTED_BASELINE) / SCALE

print("activation patching complete:", n_layers * n_heads, "separate runs")

## Method 2 - attribution patching

All heads scored at once from a single forward and backward pass.

In [ ]:
corrupted_activations = {}
corrupted_gradients = {}

def save_activation(activation, hook):
    corrupted_activations[hook.name] = activation.detach()

def save_gradient(gradient, hook):
    corrupted_gradients[hook.name] = gradient.detach()

model.reset_hooks()
for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    model.add_hook(hook_name, save_activation, "fwd")
    model.add_hook(hook_name, save_gradient, "bwd")

torch.set_grad_enabled(True)
mean_logit_difference_tensor(model(corrupted_tokens)).backward()
torch.set_grad_enabled(False)
model.reset_hooks()

attribution_scores = np.zeros((n_layers, n_heads))

for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    difference = clean_cache[hook_name] - corrupted_activations[hook_name]
    contribution = difference * corrupted_gradients[hook_name]
    for head in range(n_heads):
        attribution_scores[layer, head] = contribution[:, :, head, :].sum().item() / SCALE

print("attribution patching complete")

## Ranking the heads

Both methods produce a score for every head. Sorting from highest to lowest gives the ranked list each method would hand to a user.

In [ ]:
def rank_heads(scores):
    """Return (layer, head) pairs sorted from highest score to lowest."""
    scored = []
    for layer in range(n_layers):
        for head in range(n_heads):
            scored.append((scores[layer, head], layer, head))
    scored.sort(reverse=True)

    ranked = []
    for score, layer, head in scored:
        ranked.append((layer, head))
    return ranked

patching_rank = rank_heads(patching_scores)
attribution_rank = rank_heads(attribution_scores)

all_heads = []
for layer in range(n_layers):
    for head in range(n_heads):
        all_heads.append((layer, head))

print("activation patching top 5: ", patching_rank[:5])
print("attribution patching top 5:", attribution_rank[:5])

## Timing (RQ3)

Timing is measured separately from the calculations above, so the results are not affected.

Two details matter. Each method runs once as a warm-up before anything is recorded, and the GPU is synchronised before the clock is read. GPU work is queued rather than run immediately, so without synchronising the timer measures how long it took to queue the work instead of how long it took to finish.

In [ ]:
def run_activation_patching():
    global head_to_patch
    with torch.no_grad():
        for layer in range(n_layers):
            hook_name = utils.get_act_name("z", layer)
            for head in range(n_heads):
                head_to_patch = head
                model.run_with_hooks(corrupted_tokens, fwd_hooks=[(hook_name, patch_one_head)])

def run_attribution_patching():
    model.reset_hooks()
    for layer in range(n_layers):
        hook_name = utils.get_act_name("z", layer)
        model.add_hook(hook_name, save_activation, "fwd")
        model.add_hook(hook_name, save_gradient, "bwd")
    torch.set_grad_enabled(True)
    mean_logit_difference_tensor(model(corrupted_tokens)).backward()
    torch.set_grad_enabled(False)
    model.reset_hooks()

def time_method(function, n_repeats=3):
    """Warm up once, then time the method n_repeats times and report the mean."""
    function()
    if device == "cuda":
        torch.cuda.synchronize()

    timings = []
    for _ in range(n_repeats):
        start_time = time.time()
        function()
        if device == "cuda":
            torch.cuda.synchronize()
        timings.append(time.time() - start_time)

    return float(np.mean(timings)), float(np.std(timings))

patching_time, patching_sd = time_method(run_activation_patching)
attribution_time, attribution_sd = time_method(run_attribution_patching)

print("activation patching: ", round(patching_time, 3), "s (sd", round(patching_sd, 3), ")")
print("attribution patching:", round(attribution_time, 4), "s (sd", round(attribution_sd, 4), ")")
print("speed-up:            ", round(patching_time / attribution_time, 1), "times")

## Agreement (RQ1)

Three measures, because they answer different questions. Rank correlation asks whether the whole ordering matches. Jaccard overlap asks whether the same heads get selected at a given cut-off. Precision and recall ask whether those heads are actually in the published circuit, which is the only measure that checks against something external rather than comparing the methods to each other.

The circuit below is taken from Figure 2 of Wang et al. (2023) - 26 heads in seven functional classes.

In [ ]:
# Wang et al. (2023), Figure 2
head_roles = {
    (9, 9): "Name Mover", (9, 6): "Name Mover", (10, 0): "Name Mover",
    (10, 7): "Negative Name Mover", (11, 10): "Negative Name Mover",
    (9, 0): "Backup Name Mover", (9, 7): "Backup Name Mover",
    (10, 1): "Backup Name Mover", (10, 2): "Backup Name Mover",
    (10, 6): "Backup Name Mover", (10, 10): "Backup Name Mover",
    (11, 2): "Backup Name Mover", (11, 9): "Backup Name Mover",
    (7, 3): "S-Inhibition", (7, 9): "S-Inhibition",
    (8, 6): "S-Inhibition", (8, 10): "S-Inhibition",
    (5, 5): "Induction", (5, 8): "Induction",
    (5, 9): "Induction", (6, 9): "Induction",
    (0, 1): "Duplicate Token", (0, 10): "Duplicate Token", (3, 0): "Duplicate Token",
    (2, 2): "Previous Token", (4, 11): "Previous Token",
}

circuit_heads = list(head_roles.keys())
print("reference circuit:", len(circuit_heads), "heads")

In [ ]:
# rank correlation across all 144 heads
correlation, p_value = spearmanr(patching_scores.flatten(), attribution_scores.flatten())
print("Spearman rank correlation:", round(correlation, 3))
print()

def count_in_circuit(heads):
    count = 0
    for head in heads:
        if head in circuit_heads:
            count = count + 1
    return count

def count_shared(list_a, list_b):
    count = 0
    for head in list_a:
        if head in list_b:
            count = count + 1
    return count

print(" k   Jaccard   patch P   patch R   attr P   attr R")
for k in [5, 10, 15, 20]:
    top_patching = patching_rank[:k]
    top_attribution = attribution_rank[:k]

    shared = count_shared(top_patching, top_attribution)
    union = 2 * k - shared
    jaccard = shared / union

    patch_hits = count_in_circuit(top_patching)
    attr_hits = count_in_circuit(top_attribution)

    print(f"{k:>2}   {jaccard:>7.2f}   {patch_hits/k:>7.2f}   {patch_hits/len(circuit_heads):>7.2f}"
          f"   {attr_hits/k:>6.2f}   {attr_hits/len(circuit_heads):>6.2f}")

In [ ]:
# which functional classes appear in each method's top 20
all_classes = []
for head in circuit_heads:
    if head_roles[head] not in all_classes:
        all_classes.append(head_roles[head])

for method_name, ranking in [("activation", patching_rank), ("attribution", attribution_rank)]:
    found = []
    for head in ranking[:20]:
        if head in circuit_heads and head_roles[head] not in found:
            found.append(head_roles[head])

    missing = []
    for role in all_classes:
        if role not in found:
            missing.append(role)

    print(method_name, "top 20 covers:", sorted(found))
    print("   missing:", sorted(missing))
    print()

In [ ]:
# Figure 7.1 - both score matrices side by side, with circuit heads outlined
figure, axes = plt.subplots(1, 2, figsize=(14, 6))

panels = [(axes[0], patching_scores, "Activation patching"),
          (axes[1], attribution_scores, "Attribution patching")]

for axis, scores, title in panels:
    largest = abs(scores).max()
    image = axis.imshow(scores, cmap="RdBu", vmin=-largest, vmax=largest)

    # outline the heads that belong to the published circuit
    for layer, head in circuit_heads:
        outline = plt.Rectangle((head - 0.5, layer - 0.5), 1, 1,
                                fill=False, edgecolor="black", linewidth=1.5)
        axis.add_patch(outline)

    axis.set_title(title)
    axis.set_xlabel("head")
    axis.set_ylabel("layer")
    axis.set_xticks(range(n_heads))
    axis.set_yticks(range(n_layers))
    figure.colorbar(image, ax=axis, label="importance score")

plt.tight_layout()
plt.savefig("figure_7_1_importance_heatmaps.png", dpi=300, bbox_inches="tight")
plt.show()

## Faithfulness (RQ2)

Necessity switches off the top heads a method chose. If they mattered, the behaviour should fall away.

Sufficiency does the opposite and switches off everything else. If those heads are enough on their own, the behaviour should survive.

Both are reported as curves across a range of cut-offs rather than at one value, because the work is spread across several heads and any single cut-off would be an arbitrary choice.

In [ ]:
mean_activations = {}
for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    mean_activations[layer] = clean_cache[hook_name].mean(0, keepdim=True)

layer_of_hook = {}
for layer in range(n_layers):
    layer_of_hook[utils.get_act_name("z", layer)] = layer

heads_to_ablate = []   # set before each run

def ablate_heads(z, hook):
    layer = layer_of_hook[hook.name]
    for target_layer, target_head in heads_to_ablate:
        if target_layer == layer:
            z[:, :, target_head, :] = mean_activations[layer][:, :, target_head, :]
    return z

def ablation_hooks():
    hooks = []
    for layer in range(n_layers):
        hooks.append((utils.get_act_name("z", layer), ablate_heads))
    return hooks

def run_with_ablation(heads):
    """Mean-ablate the given heads and return the mean logit difference."""
    global heads_to_ablate
    heads_to_ablate = heads
    with torch.no_grad():
        return mean_logit_difference(model.run_with_hooks(clean_tokens, fwd_hooks=ablation_hooks()))

def run_with_ablation_per_sentence(heads):
    """Same, but one value per sentence for the paired statistical tests."""
    global heads_to_ablate
    heads_to_ablate = heads
    with torch.no_grad():
        return logit_difference_per_sentence(model.run_with_hooks(clean_tokens, fwd_hooks=ablation_hooks()))

def everything_except(heads):
    """All heads apart from the ones given - used for the sufficiency test."""
    rest = []
    for head in all_heads:
        if head not in heads:
            rest.append(head)
    return rest

In [ ]:
K_MAX = 20

necessity_patching = []
necessity_attribution = []
sufficiency_patching = []
sufficiency_attribution = []

for k in range(K_MAX + 1):
    top_patching = patching_rank[:k]
    top_attribution = attribution_rank[:k]

    necessity_patching.append(run_with_ablation(top_patching))
    necessity_attribution.append(run_with_ablation(top_attribution))
    sufficiency_patching.append(run_with_ablation(everything_except(top_patching)))
    sufficiency_attribution.append(run_with_ablation(everything_except(top_attribution)))

print("curves calculated for k = 0 to", K_MAX)

In [ ]:
# Figure 7.2 - necessity and sufficiency curves
k_values = list(range(K_MAX + 1))
figure, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(k_values, necessity_patching, "o-", label="activation patching")
axes[0].plot(k_values, necessity_attribution, "s-", label="attribution patching")
axes[0].axhline(CLEAN_BASELINE, linestyle="--", color="grey", linewidth=1)
axes[0].set_title("Necessity: top-k heads ablated")
axes[0].set_xlabel("number of heads ablated (k)")
axes[0].set_ylabel("mean logit difference")
axes[0].legend()

axes[1].plot(k_values, sufficiency_patching, "o-", label="activation patching")
axes[1].plot(k_values, sufficiency_attribution, "s-", label="attribution patching")
axes[1].axhline(CLEAN_BASELINE, linestyle="--", color="grey", linewidth=1)
axes[1].set_title("Sufficiency: only top-k heads retained")
axes[1].set_xlabel("number of heads retained (k)")
axes[1].set_ylabel("mean logit difference")
axes[1].legend()

plt.tight_layout()
plt.savefig("figure_7_2_faithfulness_curves.png", dpi=300, bbox_inches="tight")
plt.show()

### Exact values, and where the selected sets differ

Reading values off a graph is not good enough for the write-up, so they are printed here. The middle column marks the cut-offs where the two methods picked different heads.

In [ ]:
print(" k  differ   necessity (act/attr)      sufficiency (act/attr)")
for k in range(K_MAX + 1):
    shared = count_shared(patching_rank[:k], attribution_rank[:k])
    if shared == k:
        marker = "  -"
    else:
        marker = "yes"

    print(f"{k:>2}  {marker}      "
          f"{necessity_patching[k]:+.3f} / {necessity_attribution[k]:+.3f}      "
          f"{sufficiency_patching[k]:+.3f} / {sufficiency_attribution[k]:+.3f}")

## Statistical comparison

The two methods explain the same sentences, so the values are paired and the comparison is made sentence by sentence.

The Wilcoxon signed-rank test is used because it does not assume the differences are normally distributed. Confidence intervals come from bootstrap resampling, and the rank-biserial correlation gives an effect size to sit alongside the p-value.

Where both methods pick the same heads the values are identical by definition, so no test is reported.

In [ ]:
def bootstrap_confidence_interval(differences, n_samples=10000):
    """Resample the differences with replacement and take the 2.5th and 97.5th percentiles."""
    generator = np.random.default_rng(0)
    means = []
    for _ in range(n_samples):
        sample = generator.choice(differences, size=len(differences), replace=True)
        means.append(sample.mean())
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

def rank_biserial_correlation(differences):
    """Effect size for the Wilcoxon test: +1 means one method won on every sentence."""
    non_zero = differences[differences != 0]
    if len(non_zero) == 0:
        return 0.0

    ranks = rankdata(np.abs(non_zero))
    positive_total = 0.0
    negative_total = 0.0
    for i in range(len(non_zero)):
        if non_zero[i] > 0:
            positive_total = positive_total + ranks[i]
        else:
            negative_total = negative_total + ranks[i]

    return (positive_total - negative_total) / ranks.sum()

def compare(values_a, values_b, label):
    differences = values_a - values_b
    if np.allclose(differences, 0):
        print(label, "- head sets identical, values equal by definition, no test reported")
        return

    statistic, p_value = wilcoxon(values_a, values_b)
    low, high = bootstrap_confidence_interval(differences)
    effect = rank_biserial_correlation(differences)

    print(label)
    print("   activation:", round(float(values_a.mean()), 3), "| attribution:", round(float(values_b.mean()), 3))
    print("   difference:", round(float(differences.mean()), 3),
          "| 95% CI [", round(float(low), 3), ",", round(float(high), 3), "]")
    print("   Wilcoxon p =", p_value, "| rank-biserial r =", round(float(effect), 3))

In [ ]:
for k in [3, 5, 10, 20]:
    top_patching = patching_rank[:k]
    top_attribution = attribution_rank[:k]

    necessity_a = run_with_ablation_per_sentence(top_patching)
    necessity_b = run_with_ablation_per_sentence(top_attribution)
    compare(necessity_a, necessity_b, f"necessity at k = {k}")

    sufficiency_a = run_with_ablation_per_sentence(everything_except(top_patching))
    sufficiency_b = run_with_ablation_per_sentence(everything_except(top_attribution))
    compare(sufficiency_a, sufficiency_b, f"sufficiency at k = {k}")
    print()

print("Four cut-offs were pre-specified for each measure, so a Bonferroni-adjusted threshold of 0.0125 applies.")

## Which heads differ, and what they do

The faithfulness difference comes down to a small number of head substitutions. This shows which heads each method picked at the smallest cut-offs, and what role Wang et al. (2023) assign to them.

In [ ]:
def role_of(head):
    if head in head_roles:
        return head_roles[head]
    return "not in circuit"

for k in [1, 2, 3, 4, 5]:
    print("k =", k)

    print("  activation patching:")
    for layer, head in patching_rank[:k]:
        print(f"     {layer}.{head:<3} {role_of((layer, head)):<22} {patching_scores[layer, head]:+.3f}")

    print("  attribution patching:")
    for layer, head in attribution_rank[:k]:
        print(f"     {layer}.{head:<3} {role_of((layer, head)):<22} {attribution_scores[layer, head]:+.3f}")

    shared_heads = []
    for head in patching_rank[:k]:
        if head in attribution_rank[:k]:
            shared_heads.append(head)
    print("  shared:", sorted(shared_heads))
    print()

## Result

The two methods agree closely across the full ranking and select identical heads at most cut-offs, while attribution patching runs far faster. They part company at the very top of the ranking, which is the part anyone actually inspecting a model would look at.

Both figures are saved as 300 dpi PNG files for the thesis.

### References

- Meng, K. et al. (2022) *Locating and Editing Factual Associations in GPT*. NeurIPS.
- Nanda, N. (2023) *Attribution Patching: Activation Patching at Industrial Scale*.
- Nanda, N. and Bloom, J. (2022) *TransformerLens*.
- Wang, K. et al. (2023) *Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small*. ICLR.